# Analyze Eclipse PVT data

In [ ]:
# Import Python libraries
!pip install -q kaleido==0.2.1 -q skimpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.0/118.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.5 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: jupyter-client
    Found existing installation: jupyter_client 7.4.9
    Uninstalling jupyter_client-7.4.9:
      Successfully uninstalled jupyter_client-7.4.9
  Attempting uninstall: ipykerne

In [ ]:
import os
from google.colab import drive
from pathlib import Path

# Mount Google Drive (run once per session)
drive.mount("/content/drive")

# Data directory (adjust if you move the notebook)
source = Path("/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PVT/")
dest = Path("/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Output/Norne/")

# Create directory if it does not exist
os.makedirs(dest, exist_ok=True)

print("Source directory:", source)
print("Output directory:", dest)

Mounted at /content/drive
Source directory: /content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PVT
Output directory: /content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Output/Norne


In [ ]:
from pathlib import Path

# All entries (files + subdirs)
for p in source.iterdir():
    print(p)

# Only files
#for p in source.iterdir():
#    if p.is_file():
#        print(p)

/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PVT/PVT-WET-GAS.INC


## 1.0 Typical wet‑gas PVT prep workflow ##

For a black‑oil wet‑gas case like this, a practical Python workflow is:​

1. Parse PVTG (gas‑phase table) into a structured dataframe: pressure, solution gas in oil Rsg, gas FVF Bg, gas viscosity.​

2. Parse PVTO (undersaturated/volumetric oil table) into a dataframe:
Rso, pressure, oil FVF Bo, oil viscosity.​

3. Run consistency checks (monotonicity vs P, reasonable ranges, duplicates, region splits).​

4. Optionally: interpolate to your simulation pressure grid, smooth small kinks, and add extrapolated points only where the simulator needs them.

5. Export back to clean INCLUDE sections (same Eclipse keywords, re‑formatted) or to another simulator’s format (e.g., tNavigator, CMG) with unit conversion if needed.

## 2.0 Parsing these PVT blocks with Python

The snippet below shows a robust pattern to go from the INC file to tidy PVTG and PVTO dataframes; we can adapt the parsing function style to other tables (e.g., PVDG, PVTW).

In [ ]:
import re
import pandas as pd
from pathlib import Path

inc_path = Path(source / "PVT-WET-GAS.INC")
text = inc_path.read_text()

def extract_block(keyword, text, stop_keywords=("PVTO", "PVDG", "PVCDO", "/\n/")):
    # start at keyword
    start = text.index(keyword)
    after = text[start + len(keyword):]

    # find first stop keyword after start
    stop_positions = []
    for kw in stop_keywords:
        try:
            pos = after.index(kw)
            stop_positions.append(pos)
        except ValueError:
            pass
    stop = min(stop_positions) if stop_positions else len(after)
    return after[:stop]

pvtg_raw = extract_block("PVTG", text, stop_keywords=("PVTO",))
pvto_raw = extract_block("PVTO", text, stop_keywords=("PVTG","PVDG","PVTW"))

def lines_without_comments(block):
    out = []
    for ln in block.splitlines():
        # strip Eclipse comment markers
        if "--" in ln:
            ln = ln.split("--",1)[0]
        ln = ln.rstrip()
        if ln:
            out.append(ln)
    return out

pvtg_lines = lines_without_comments(pvtg_raw)
pvto_lines = lines_without_comments(pvto_raw)

In [ ]:
pvtg_lines

['     50.00    0.00000497   0.024958     0.01441',
 '              0.00000248   0.024958     0.01440',
 '              0.00000000   0.024958     0.01440 /',
 '     70.00    0.00000521   0.017639     0.01491',
 '              0.00000261   0.017641     0.01490',
 '              0.00000000   0.017643     0.01490 /',
 '     90.00    0.00000627   0.013608     0.01547',
 '              0.00000313   0.013611     0.01546',
 '              0.00000000   0.013615     0.01544 /',
 '    110.00    0.00000798   0.011072     0.01609',
 '              0.00000399   0.011076     0.01607',
 '              0.00000000   0.011081     0.01605 /',
 '    130.00    0.00001041   0.009340     0.01677',
 '              0.00000520   0.009346     0.01674',
 '              0.00000000   0.009352     0.01671 /',
 '    150.00    0.00001365   0.008092     0.01752',
 '              0.00000683   0.008099     0.01748',
 '              0.00000000   0.008106     0.01743 /',
 '    170.00    0.00001786   0.007156     0.01834',


Now interpret the PVTG structure (each pressure has 3 lines: two Rs at intermediate points and one Rs = 0 line with “/” at end).

In [ ]:
def parse_pvtg(lines):
    records = []
    i = 0
    while i < len(lines):
        # first line always starts with pressure
        parts = lines[i].split()
        if len(parts) < 4:
            i += 1
            continue
        p = float(parts[0])
        rsg = float(parts[1])
        bg  = float(parts[2])
        mu  = float(parts[3])
        records.append({"P_bar": p, "Rsg_m3m3": rsg, "Bg_rm3sm3": bg, "mu_gas_cP": mu})
        i += 1

        # next one or two continuation lines (no pressure)
        while i < len(lines):
            parts = lines[i].split()
            # last line has "/" at end
            if parts[-1].endswith("/"):
                # strip "/"
                parts[-1] = parts[-1].replace("/", "")
                rsg = float(parts[0])
                bg  = float(parts[1])
                mu  = float(parts[2])
                records.append({"P_bar": p, "Rsg_m3m3": rsg, "Bg_rm3sm3": bg, "mu_gas_cP": mu})
                i += 1
                break
            else:
                # continuation
                rsg = float(parts[0])
                bg  = float(parts[1])
                mu  = float(parts[2])
                records.append({"P_bar": p, "Rsg_m3m3": rsg, "Bg_rm3sm3": bg, "mu_gas_cP": mu})
                i += 1
    return pd.DataFrame(records)

pvtg_df = parse_pvtg(pvtg_lines)
print(pvtg_df.head())

   P_bar  Rsg_m3m3  Bg_rm3sm3  mu_gas_cP
0   50.0  0.000005   0.024958    0.01441
1   50.0  0.000002   0.024958    0.01440
2   50.0  0.000000   0.024958    0.01440
3   70.0  0.000005   0.017639    0.01491
4   70.0  0.000003   0.017641    0.01490


In [ ]:
from skimpy import skim

skim(pvtg_df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 147    │ │ float64     │ 4     │                                                          │
│ │ Number of columns │ 4      │ └─────────────┴───────┘                                                          │
│ └───────────────────┴────────┘                                                                                  │
│                                                     number                                                      │
│ ┏━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┓  │
│ ┃ column   ┃ NA ┃ NA % ┃ mean     ┃ sd       ┃ p0       ┃ p25      ┃ p50      ┃ p75      ┃ p100     ┃ hist   ┃  │
│ ┡━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━┩  │
│ │ P_bar    │  0 │    0 │    334.7 │    169.3 │       50 │      180 │    346.8 │    488.3 │    594.3 │ ▇▇▄▅▆█ │  │
│ │ Rsg_m3m3 │  0 │    0 │ 0.000107 │ 0.000175 │        0 │        0 │ 1.365e-0 │ 0.000152 │ 0.000825 │  █▁▁   │  │
│ │          │    │      │        8 │        1 │          │          │        5 │        5 │        9 │        │  │
│ │ Bg_rm3sm │  0 │    0 │  0.00594 │ 0.004388 │ 0.002828 │ 0.003306 │ 0.003911 │ 0.006683 │  0.02496 │  █▂▁   │  │
│ │ 3        │    │      │          │          │          │          │          │          │          │        │  │
│ │ mu_gas_c │  0 │    0 │  0.02918 │   0.0129 │   0.0144 │  0.01859 │  0.02703 │  0.03488 │  0.07567 │  █▆▂▁  │  │
│ │ P        │    │      │          │          │          │          │          │          │          │        │  │
│ └──────────┴────┴──────┴──────────┴──────────┴──────────┴──────────┴──────────┴──────────┴──────────┴────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯

In [ ]:
import plotly.express as px

# Gas compressibility (Bg vs pressure)
fig_bg = px.scatter(
    pvtg_df.sort_values("P_bar"),
    x="P_bar", y="Bg_rm3sm3",
    title="Gas FVF vs pressure (PVTG)",
    labels={"P_bar": "Pressure (bar)", "Bg_rm3sm3": "Bg (rm3/sm3)"}
)

fig_bg.update_layout(
    template='ggplot2',
    width=900,    # reduced width
    height=600,   # optional
)

fig_bg.show()

# Gas viscosity vs pressure
fig_mug = px.scatter(
    pvtg_df.sort_values("P_bar"),
    x="P_bar", y="mu_gas_cP",
    title="Gas viscosity vs pressure (PVTG)",
    labels={"P_bar": "Pressure (bar)", "mu_g_cp": "Gas viscosity (cp)"}
)

fig_mug.update_layout(
    template='ggplot2',
    width=900,    # reduced width
    height=600,   # optional
)

fig_mug.show()

For PVTO, each block starts with an Rso line followed by several pressure rows, terminated by “/”.

In [ ]:
def parse_pvto(lines):
    records = []
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        # identify start of new Rso block: first token is Rso, second is pressure
        if len(parts) >= 4:
            try:
                rso = float(parts[0])
                p   = float(parts[1])
                bo  = float(parts[2])
                mu  = float(parts[3])
            except ValueError:
                i += 1
                continue
            current_rso = rso
            records.append({"Rso_m3m3": rso, "P_bar": p, "Bo_rm3sm3": bo, "mu_oil_cP": mu})
            i += 1
        else:
            i += 1
            continue

        # continuation rows for same Rso
        while i < len(lines):
            parts = lines[i].split()
            # line ending with "/" closes this Rso block
            if parts[-1].endswith("/"):
                parts[-1] = parts[-1].replace("/", "")
                p   = float(parts[0])
                bo  = float(parts[1])
                mu  = float(parts[2])
                records.append({"Rso_m3m3": current_rso, "P_bar": p, "Bo_rm3sm3": bo, "mu_oil_cP": mu})
                i += 1
                break
            else:
                p   = float(parts[0])
                bo  = float(parts[1])
                mu  = float(parts[2])
                records.append({"Rso_m3m3": current_rso, "P_bar": p, "Bo_rm3sm3": bo, "mu_oil_cP": mu})
                i += 1
    return pd.DataFrame(records)

pvto_df = parse_pvto(pvto_lines)
print(pvto_df.head())

   Rso_m3m3  P_bar  Bo_rm3sm3  mu_oil_cP
0     20.59   50.0    1.10615      1.180
1     20.59   75.0    1.10164      1.247
2     20.59  100.0    1.09744      1.315
3     20.59  125.0    1.09351      1.384
4     20.59  150.0    1.08984      1.453


In [ ]:
from skimpy import skim

skim(pvto_df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 237    │ │ float64     │ 4     │                                                          │
│ │ Number of columns │ 4      │ └─────────────┴───────┘                                                          │
│ └───────────────────┴────────┘                                                                                  │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━┳━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓  │
│ ┃ column      ┃ NA  ┃ NA %  ┃ mean     ┃ sd       ┃ p0       ┃ p25     ┃ p50     ┃ p75     ┃ p100   ┃ hist   ┃  │
│ ┡━━━━━━━━━━━━━╇━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩  │
│ │ Rso_m3m3    │   0 │     0 │    189.5 │    118.9 │    20.59 │    79.5 │   179.6 │   297.5 │  404.6 │ █▅▄▄▄▄ │  │
│ │ P_bar       │   0 │     0 │    391.1 │    171.8 │       50 │     235 │   407.3 │     543 │  694.3 │ ▄▇▅▆█▅ │  │
│ │ Bo_rm3sm3   │   0 │     0 │    1.482 │    0.265 │     1.09 │   1.232 │   1.458 │   1.713 │  1.975 │ █▅▄▄▄▄ │  │
│ │ mu_oil_cP   │   0 │     0 │   0.5426 │   0.3034 │   0.2156 │  0.2841 │  0.4169 │  0.7699 │  1.453 │ █▂▃▂▁  │  │
│ └─────────────┴─────┴───────┴──────────┴──────────┴──────────┴─────────┴─────────┴─────────┴────────┴────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯

At this point we have clean PVT tables in Pandas suitable for:

1. building interpolation functions (e.g., SciPy interp1d, RegularGridInterpolator),

2. plotting consistency checks,

3. sampling onto a simulator’s pressure schedule or regions, and

4. exporting to alternate formats.

## 3.0 Visualize key PVT relationships

In [ ]:
import plotly.express as px

# Gas FVF vs pressure by Rsg
fig_bg = px.line(
    pvtg_df.sort_values(["Rsg_m3m3", "P_bar"]),
    x="P_bar", y="Bg_rm3sm3", color="Rsg_m3m3",
    title="Gas FVF (Bg) vs pressure",
    labels={"P_bar": "Pressure (bar)", "Bg_rm3sm3": "Bg (rm3/sm3)", "Rsg_m3m3": "Rsg (m3/m3)"},
)

fig_bg.update_layout(
    template='ggplot2',
    width=900,    # reduced width
    height=600,   # optional
)

fig_bg.show()

# Oil Bo vs pressure colored by Rso
fig_bo = px.line(
    pvto_df.sort_values(["Rso_m3m3", "P_bar"]),
    x="P_bar", y="Bo_rm3sm3", color="Rso_m3m3",
    title="Oil FVF (Bo) vs pressure, colored by Rso",
    labels={"P_bar": "Pressure (bar)", "Bo_rm3sm3": "Bo (rm3/sm3)", "Rso_m3m3": "Rso (sm3/sm3)"},
)

fig_bo.update_layout(
    template='ggplot2',
    width=900,    # reduced width
    height=600,   # optional
)
fig_bo.show()

# Oil viscosity vs pressure
fig_mu_o = px.line(
    pvto_df.sort_values(["Rso_m3m3", "P_bar"]),
    x="P_bar", y="mu_oil_cP", color="Rso_m3m3",
    title="Oil viscosity vs pressure, colored by Rso",
    labels={"P_bar": "Pressure (bar)", "mu_oil_cP": "Oil viscosity (cp)", "Rso_m3m3": "Rso (m3/m3)"},
)

fig_mu_o.update_layout(
    template='ggplot2',
    width=900,    # reduced width
    height=600,   # optional
)

fig_mu_o.show()

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# --- 1. Approximate oil compressibility from PVTO ---

def add_oil_compressibility(df_pvto):
    # sort for stable differencing
    df = df_pvto.sort_values(["Rso_m3m3", "P_bar"]).copy() # Corrected column name
    df["c_o_1bar"] = np.nan

    for rso_val, grp in df.groupby(["Rso_m3m3"]): # Removed 'region' and used correct column name
        p = grp["P_bar"].values
        bo = grp["Bo_rm3sm3"].values # Corrected column name

        # central differences for internal points, one-sided on ends
        dBo_dP = np.gradient(bo, p)
        c_o = -1.0 / bo * dBo_dP  # 1/bar

        df.loc[grp.index, "c_o_1bar"] = c_o

    return df

df_pvto_c = add_oil_compressibility(pvto_df)

fig_co = px.line(
    df_pvto_c,
    x="P_bar",
    y="c_o_1bar",
    color="Rso_m3m3", # Corrected column name
    # facet_row="region", # Removed as 'region' column does not exist
    title="Approximate oil compressibility vs pressure",
    labels={"P_bar": "Pressure (bar)", "c_o_1bar": "c_o (1/bar)", "Rso_m3m3": "Rso (sm3/sm3)"}, # Corrected label
)
fig_co.update_yaxes(matches=None)

fig_co.update_layout(
    template='ggplot2',
    width=900,    # reduced width
    height=600,   # optional
)

fig_co.show()

In [ ]:
# --- 2. Identify bubblepoint per Rso sequence (simple heuristic) ---

def find_bubblepoints(pvto_df):
    # Sort by Rso and Pressure. 'region' column does not exist in pvto_df.
    df = pvto_df.sort_values(["Rso_m3m3", "P_bar"]).copy()
    bps = []

    # Group by Rso. 'region' column does not exist.
    for rso, grp in df.groupby(["Rso_m3m3"]):
        # crude heuristic: bubblepoint near maximum Bo
        # Use actual column names 'Bo_rm3sm3' and 'P_bar'
        idx_max = grp["Bo_rm3sm3"].idxmax()
        row = grp.loc[idx_max]
        bps.append(
            {
                "Rso_m3m3": rso, # Corrected column name
                "P_bub_bar": row["P_bar"],
                "Bo_at_bub": row["Bo_rm3sm3"], # Corrected column name
            }
        )
    return pd.DataFrame(bps)

df_bub = find_bubblepoints(pvto_df)

fig_bub = px.scatter(
    df_bub,
    x="P_bub_bar",
    y="Bo_at_bub",
    color="Rso_m3m3", # Corrected column name
    # facet_row="region", # Removed as 'region' column does not exist
    title="Estimated bubblepoint (Bo peak) per Rso",
    labels={
        "P_bub_bar": "Bubblepoint pressure (bar)",
        "Bo_at_bub": "Bo at bubblepoint",
        "Rso_m3m3": "Rso (sm3/sm3)" # Corrected label for Rso
    },
)

fig_bub.update_layout(
    template='ggplot2',
    width=900,    # reduced width
    height=600,   # optional
)

fig_bub.show()

In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

deg = 3  # polynomial degree

x = df_bub["P_bub_bar"].to_numpy()
y = df_bub["Bo_at_bub"].to_numpy()

coeffs = np.polyfit(x, y, deg)
poly = np.poly1d(coeffs)

x_fit = np.linspace(x.min(), x.max(), 200)
y_fit = poly(x_fit)

# scatter + markers
fig_bub = px.scatter(
    df_bub,
    x="P_bub_bar",
    y="Bo_at_bub",
    title="Estimated bubblepoint (Bo peak)",
)

# best-fit line (keeps legend entry)
fig_bub.add_trace(
    go.Scatter(
        x=x_fit,
        y=y_fit,
        mode="lines",
        name=f"Poly fit (deg {deg})",
        line=dict(color="red"),
    )
)

# equation string
terms = []
p = deg
for c in coeffs:
    if p > 1:
        terms.append(f"{c:.4g}·P^{p}")
    elif p == 1:
        terms.append(f"{c:.4g}·P")
    else:
        terms.append(f"{c:.4g}")
    p -= 1
eq_str = "Bo = " + " + ".join(terms)

# add annotation inside plot area
fig_bub.add_annotation(
    xref="paper",
    yref="paper",
    x=0.05,
    y=0.95,
    text=eq_str,
    showarrow=False,
    bordercolor="black",
    borderwidth=1,
    bgcolor="white",
    font=dict(size=12),
)

fig_bub.update_layout(
    template="ggplot2",
    width=900,
    height=600,
)

fig_bub.show()

NameError: name 'df_bub' is not defined

In [ ]:
# --- 3. Oil mobility vs pressure (k/mu_o), assuming some reference k ---

# Choose a reference permeability, say 100 mD
k_ref_md = 100.0

df_pvto_mob = pvto_df.copy()
df_pvto_mob["mob_o_mDcp"] = k_ref_md / df_pvto_mob["mu_oil_cP"]

fig_mob = px.line(
    df_pvto_mob.sort_values(["Rso_m3m3", "P_bar"]),
    x="P_bar",
    y="mob_o_mDcp",
    color="Rso_m3m3",
    title="Oil mobility (k/μ) vs pressure at reference k",
    labels={"P_bar": "Pressure (bar)", "mob_o_mDcp": "Oil mobility (mD/cp)", "Rso_m3m3": "Rso"},
)
fig_mob.update_yaxes(matches=None)

fig_mob.update_layout(
    template='ggplot2',
    width=900,    # reduced width
    height=600,   # optional
)

fig_mob.show()

## 4.0 Exporting back to simulator input ##

To regenerate an INCLUDE with cleaned or re‑sampled PVTG/PVTO:

In [ ]:
def format_pvtg(df):
    # group by pressure, write 3 rows like the original
    lines = ["PVTG"]
    for p, grp in df.groupby("P_bar"):
        grp = grp.sort_values("Rsg_m3m3")
        for j, (_, row) in enumerate(grp.iterrows()):
            rsg = f"{row.Rsg_m3m3: .8f}"
            bg  = f"{row.Bg_rm3sm3: .6f}"
            mu  = f"{row.mu_gas_cP: .5f}"
            if j == 0:
                lines.append(f"{p:10.2f}{rsg:>14}{bg:>11}{mu:>11}")
            elif j == len(grp)-1:
                lines.append(f"{'':10s}{rsg:>14}{bg:>11}{mu:>11} /")
            else:
                lines.append(f"{'':10s}{rsg:>14}{bg:>11}{mu:>11}")
    lines.append("/")
    return "\n".join(lines)

def format_pvto(df):
    lines = ["PVTO"]
    for rso, grp in df.groupby("Rso_m3m3"):
        grp = grp.sort_values("P_bar")
        first = True
        for _, row in grp.iterrows():
            p  = f"{row.P_bar: .2f}"
            bo = f"{row.Bo_rm3sm3: .5f}"
            mu = f"{row.mu_oil_cP: .3f}"
            if first:
                lines.append(f"{rso:8.2f}{p:10}{bo:>11}{mu:>8}")
                first = False
            else:
                lines.append(f"{'':8s}{p:10}{bo:>11}{mu:>8}")
        # close block
        lines[-1] = lines[-1] + "  /"
    lines.append("/")
    return "\n".join(lines)

new_pvtg_inc = format_pvtg(pvtg_df)
new_pvto_inc = format_pvto(pvto_df)

Path("PVT-WET-GAS-CLEAN-PVTG.INC").write_text(new_pvtg_inc)
Path("PVT-WET-GAS-CLEAN-PVTO.INC").write_text(new_pvto_inc)

9159

## 5.0 Interpolation to a target pressure grid ##

Assuming we already have pvtg_df and pvto_df from the earlier parsing step, with columns as in the previous reply.

In [ ]:
import numpy as np
from scipy.interpolate import interp1d

# Example: define your simulation pressure grid (bar)
p_grid = np.linspace(50, 600, 40)   # change to your schedule / PVT region range

def interp_pvtg_to_grid(pvtg_df, p_grid):
    """Interpolate PVTG to a given pressure grid at each Rsg."""
    out_records = []
    for rsg, grp in pvtg_df.groupby("Rsg_m3m3"):
        grp = grp.sort_values("P_bar")
        f_bg = interp1d(grp["P_bar"], grp["Bg_rm3sm3"], kind="linear", fill_value="extrapolate")
        f_mu = interp1d(grp["P_bar"], grp["mu_gas_cP"],   kind="linear", fill_value="extrapolate")
        for p in p_grid:
            out_records.append({
                "P_bar": p,
                "Rsg_m3m3": rsg,
                "Bg_rm3sm3": float(f_bg(p)),
                "mu_gas_cP": float(f_mu(p))
            })
    return pd.DataFrame(out_records)

def interp_pvto_to_grid(pvto_df, p_grid):
    """Interpolate PVTO to a given pressure grid at each Rso."""
    out_records = []
    for rso, grp in pvto_df.groupby("Rso_m3m3"):
        grp = grp.sort_values("P_bar")
        f_bo = interp1d(grp["P_bar"], grp["Bo_rm3sm3"], kind="linear", fill_value="extrapolate")
        f_mu = interp1d(grp["P_bar"], grp["mu_oil_cP"], kind="linear", fill_value="extrapolate")
        for p in p_grid:
            # only keep target pressures within original range if you want to avoid extrapolation:
            if p < grp["P_bar"].min() or p > grp["P_bar"].max():
                continue
            out_records.append({
                "Rso_m3m3": rso,
                "P_bar": p,
                "Bo_rm3sm3": float(f_bo(p)),
                "mu_oil_cP": float(f_mu(p))
            })
    return pd.DataFrame(out_records)

pvtg_interp = interp_pvtg_to_grid(pvtg_df, p_grid)
pvto_interp = interp_pvto_to_grid(pvto_df, p_grid)

/usr/local/lib/python3.12/dist-packages/scipy/interpolate/_interpolate.py:497: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]


You can feed pvtg_interp and pvto_interp into the formatting functions from the previous step to regenerate INCLUDE decks on your preferred pressure grid.

## 6.0 Detecting non‑monotone behaviour ##

A quick monotonicity check is easy with diff() sign changes. Below, “non‑monotone” means a sign flip in the first differences vs pressure at fixed
Rsg or Rso.

In [ ]:
def find_non_monotone(df, x_col, y_col, group_cols=None, direction="increasing"):
    """
    direction: 'increasing' or 'decreasing'
    Returns rows where the monotonicity condition is violated within each group.
    """
    if group_cols is None:
        group_cols = []

    viol_idx = []
    for _, grp in df.sort_values(x_col).groupby(group_cols):
        y = grp[y_col].values
        dy = np.diff(y)
        if direction == "increasing":
            bad = np.where(dy < 0)[0]
        else:
            bad = np.where(dy > 0)[0]
        if len(bad):
            viol_idx.extend(grp.iloc[bad + 1].index.tolist())
    return df.loc[viol_idx]

# examples:
viol_bg  = find_non_monotone(pvtg_df, "P_bar", "Bg_rm3sm3",
                             group_cols=["Rsg_m3m3"], direction="increasing")
viol_mug = find_non_monotone(pvtg_df, "P_bar", "mu_gas_cP",
                             group_cols=["Rsg_m3m3"], direction="increasing")
viol_bo  = find_non_monotone(pvto_df, "P_bar", "Bo_rm3sm3",
                             group_cols=["Rso_m3m3"], direction="decreasing")
viol_muo = find_non_monotone(pvto_df, "P_bar", "mu_oil_cP",
                             group_cols=["Rso_m3m3"], direction="increasing")

print("Non‑monotone Bg rows:", len(viol_bg))
print("Non‑monotone Bo rows:", len(viol_bo))

Non‑monotone Bg rows: 48
Non‑monotone Bo rows: 0


## 7.0 Simple smoothing that enforces monotonicity ##

For PVT prep the usual goal is “physically reasonable and numerically stable”, not fancy fitting, so a pragmatic approach is:

1. median or Savitzky–Golay smoothing to remove high‑frequency noise, then

2. projection back to a monotone sequence via isotonic regression (piecewise constant / linear).

Below is a light isotonic‑style “enforce monotone” pass using NumPy only.

In [ ]:
def enforce_monotone(y, direction="increasing"):
    """Return a copy of y forced to be monotone (pool-adjacent-violators)."""
    y = np.asarray(y, dtype=float)
    if direction == "decreasing":
        y_work = -y.copy()
    else:
        y_work = y.copy()

    # pool-adjacent-violators algorithm
    n = len(y_work)
    g = y_work.copy()
    w = np.ones(n)
    i = 0
    while i < n - 1:
        if g[i] > g[i+1]:
            # violation: pool i and i+1 (and possibly backward)
            j = i
            total_w = w[j] + w[j+1]
            avg = (g[j]*w[j] + g[j+1]*w[j+1]) / total_w
            g[j] = g[j+1] = avg
            w[j] = w[j+1] = total_w
            # walk back
            while j > 0 and g[j-1] > g[j]:
                total_w = w[j-1] + w[j]
                avg = (g[j-1]*w[j-1] + g[j]*w[j]) / total_w
                g[j-1] = g[j] = avg
                w[j-1] = w[j] = total_w
                j -= 1
            i = j
        else:
            i += 1

    if direction == "decreasing":
        return -g
    else:
        return g

Apply this within each PVT group:

In [ ]:
def smooth_pvtg_monotone(pvtg_df):
    out = []
    for rsg, grp in pvtg_df.groupby("Rsg_m3m3"):
        grp = grp.sort_values("P_bar").copy()
        # Bg usually increasing with P in compressed gas tables; adjust if needed
        grp["Bg_rm3sm3"]  = enforce_monotone(grp["Bg_rm3sm3"].values,
                                             direction="increasing")
        grp["mu_gas_cP"]  = enforce_monotone(grp["mu_gas_cP"].values,
                                             direction="increasing")
        out.append(grp)
    return pd.concat(out, ignore_index=True)

def smooth_pvto_monotone(pvto_df):
    out = []
    for rso, grp in pvto_df.groupby("Rso_m3m3"):
        grp = grp.sort_values("P_bar").copy()
        # Undersaturated oil Bo typically decreases with pressure above Pb
        grp["Bo_rm3sm3"]  = enforce_monotone(grp["Bo_rm3sm3"].values,
                                             direction="decreasing")
        grp["mu_oil_cP"]  = enforce_monotone(grp["mu_oil_cP"].values,
                                             direction="increasing")
        out.append(grp)
    return pd.concat(out, ignore_index=True)

pvtg_smooth = smooth_pvtg_monotone(pvtg_interp)   # or pvtg_df
pvto_smooth = smooth_pvto_monotone(pvto_interp)   # or pvto_df

In [ ]:
pvtg_smooth

,P_bar,Rsg_m3m3,Bg_rm3sm3,mu_gas_cP
0,50.000000,0.000000,0.020779,0.014400
1,64.102564,0.000000,0.020779,0.014753
2,78.205128,0.000000,0.020779,0.015015
3,92.307692,0.000000,0.020779,0.015484
4,106.410256,0.000000,0.020779,0.015899
...,...,...,...,...
3955,543.589744,0.000826,NaN,NaN
3956,557.692308,0.000826,NaN,NaN
3957,571.794872,0.000826,NaN,NaN
3958,585.897436,0.000826,NaN,NaN


In [ ]:
skim(pvtg_smooth)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 3960   │ │ float64     │ 4     │                                                          │
│ │ Number of columns │ 4      │ └─────────────┴───────┘                                                          │
│ └───────────────────┴────────┘                                                                                  │
│                                                     number                                                      │
│ ┏━━━━━━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┓  │
│ ┃ column  ┃ NA   ┃ NA %    ┃ mean    ┃ sd       ┃ p0      ┃ p25     ┃ p50      ┃ p75     ┃ p100     ┃ hist   ┃  │
│ ┡━━━━━━━━━╇━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━┩  │
│ │ P_bar   │    0 │       0 │     325 │    162.8 │      50 │   187.5 │      325 │   462.5 │      600 │ ██▇█▇█ │  │
│ │ Rsg_m3m │    0 │       0 │ 0.00016 │ 0.000192 │       0 │ 1.247e- │ 8.041e-0 │ 0.00024 │ 0.000825 │  █▂▁▁  │  │
│ │ 3       │      │         │      01 │          │         │      05 │        5 │      98 │        9 │        │  │
│ │ Bg_rm3s │ 3920 │ 98.9898 │ 0.02078 │ 1.757e-1 │ 0.02078 │ 0.02078 │  0.02078 │ 0.02078 │  0.02078 │    █   │  │
│ │ m3      │      │ 9898989 │         │        7 │         │         │          │         │          │        │  │
│ │         │      │     899 │         │          │         │         │          │         │          │        │  │
│ │ mu_gas_ │ 3920 │ 98.9898 │ 0.02484 │ 0.006733 │  0.0144 │ 0.01889 │   0.0249 │ 0.03063 │  0.03581 │ █▅▄▅▆▆ │  │
│ │ cP      │      │ 9898989 │         │          │         │         │          │         │          │        │  │
│ │         │      │     899 │         │          │         │         │          │         │          │        │  │
│ └─────────┴──────┴─────────┴─────────┴──────────┴─────────┴─────────┴──────────┴─────────┴──────────┴────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯

In [ ]:
skim(pvto_smooth)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 316    │ │ float64     │ 4     │                                                          │
│ │ Number of columns │ 4      │ └─────────────┴───────┘                                                          │
│ └───────────────────┴────────┘                                                                                  │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━┳━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓  │
│ ┃ column      ┃ NA  ┃ NA %  ┃ mean     ┃ sd       ┃ p0       ┃ p25     ┃ p50     ┃ p75     ┃ p100   ┃ hist   ┃  │
│ ┡━━━━━━━━━━━━━╇━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩  │
│ │ Rso_m3m3    │   0 │     0 │    162.4 │    105.3 │    20.59 │   67.04 │   140.1 │   248.5 │  404.6 │ █▄▄▃▄▁ │  │
│ │ P_bar       │   0 │     0 │    354.3 │    156.4 │       50 │   219.2 │   360.3 │   501.3 │    600 │ ▃█▅▅▆█ │  │
│ │ Bo_rm3sm3   │   0 │     0 │    1.423 │   0.2374 │     1.09 │    1.21 │   1.385 │   1.621 │  1.972 │ █▅▄▄▄▁ │  │
│ │ mu_oil_cP   │   0 │     0 │   0.5938 │   0.3031 │   0.2166 │  0.3214 │  0.5075 │  0.8203 │  1.449 │ █▃▄▃▁  │  │
│ └─────────────┴─────┴───────┴──────────┴──────────┴──────────┴─────────┴─────────┴─────────┴────────┴────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯